In [31]:
# 从 sklearn 特征提取模块导入 DictVectorizer，用于将字典列表转为数值特征矩阵
from sklearn.feature_extraction import DictVectorizer  # 类：类别键 one-hot 展开，数值键保留原值
# 导入词袋模型 CountVectorizer 和 TF-IDF 权重向量化器 TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer  # CountVectorizer返回整数词频；TfidfVectorizer返回浮点TF-IDF权重
# 导入归一化器 MinMaxScaler 和标准化器 StandardScaler
from sklearn.preprocessing import MinMaxScaler, StandardScaler  # MinMaxScaler缩放到[0,1]；StandardScaler零均值单位方差
# 导入方差阈值特征选择器，用于过滤低方差（几乎不变）的特征列
from sklearn.feature_selection import VarianceThreshold  # fit_transform返回删除低方差列后的ndarray
# 导入主成分分析降维工具 PCA
from sklearn.decomposition import PCA  # n_components控制保留的主成分数量或累计方差比例
import jieba  # 中文分词库；cut()返回词语迭代器，lcut()返回列表
import numpy as np  # 数值计算基础库；矩阵用ndarray表示，形状为(行, 列)
from sklearn.impute import SimpleImputer  # 缺失值填充器；strategy参数可选mean/median/constant/most_frequent
import pandas as pd  # 数据处理库；DataFrame是二维表格数据结构，支持多种数据类型


# 第1节 DictVectorizer —— 字典特征向量化

**作用**：将 `list[dict]` 格式的样本集转换为数值特征矩阵。

| 特征类型 | 处理方式 | 示例 |
|---------|---------|------|
| 类别型（值为字符串） | One-Hot 展开，列名格式为「键=值」 | `城市=北京` |
| 数值型（值为 int/float） | 直接保留原始数值，列名即键名 | `温度` |

**核心参数**：
- `sparse`（bool）：`True` 返回 `scipy.csr_matrix`（节省内存）；`False` 返回 `numpy.ndarray`（便于调试）

**核心方法**：
- `fit_transform(X)`：学习词汇表并转换，返回形状 `(n_samples, n_features)` 的矩阵
- `get_feature_names_out()`：返回 `ndarray[str]`，顺序与矩阵列对应
- `inverse_transform(X)`：将数值矩阵还原回 `list[dict]`


In [32]:
# DictVectorizer 示例：将字典列表转为混合特征矩阵（类别特征 one-hot，数值特征保留）

# 构造字典列表；每个 dict 代表一个样本，键为特征名，值为特征值（类型：list[dict]）
data = [  # 3 个样本组成的列表，每个 dict 含城市（类别）、温度/湿度（数值）、风力（类别）
    {'城市': '北京', '温度': 100, '湿度': 60, '风力': '强'},  # 样本1：类别键值将被 one-hot 展开
    {'城市': '上海', '温度': 90, '湿度': 70, '风力': '中'},   # 样本2
    {'城市': '深圳', '温度': 110, '湿度': 80, '风力': '弱'}  # 样本3
]  # 列表右括号

# sparse=False 让 fit_transform 返回 numpy.ndarray 而非 scipy 稀疏矩阵，便于直接打印
dict_vec = DictVectorizer(sparse=False)  # 初始化向量化器；sparse=True时返回csr_matrix节省内存

# fit_transform：先学习所有键/值组成的词汇表，再将 data 转为 (n_samples, n_features) 的 ndarray
feature_matrix = dict_vec.fit_transform(data)  # 返回值：ndarray，形状 (3, 8)；每列对应一个特征

# get_feature_names_out()：返回 ndarray[str]，顺序与矩阵列对应；类别特征形如「城市=北京」
feature_names = dict_vec.get_feature_names_out()  # 返回值：ndarray[str]，长度等于特征矩阵列数
# 将特征矩阵转换为DataFrame，列索引为feature_names
feature_df = pd.DataFrame(feature_matrix, columns=feature_names)  # 将 ndarray 转为 DataFrame；行=样本，列=特征名（如「城市=北京」）
print("特征DataFrame:\n", feature_df)  # 打印特征矩阵 DataFrame；行=样本序号，列=特征名，值=0/1 或原始数值

print("特征名称:", feature_names)  # 查看自动生成的特征列名；数值键保留原名，类别键展开为「键=值」
print("特征矩阵形状:", feature_matrix.shape)  # 返回 tuple，如 (3, 8) 表示 3 个样本、8 个特征
print('特征类型', type(feature_matrix))  # 应输出 <class 'numpy.ndarray'>

# inverse_transform：将数值矩阵还原回字典列表；类别特征以「键=值」形式出现，数值为 float
original_data = dict_vec.inverse_transform(feature_matrix)  # 返回值：list[dict]，长度等于样本数
print("\n转换回的字典数据:")  # 打印标题
for item in original_data:  # 遍历还原后的每个样本字典；item 类型为 dict
    # 值的实际类型为 numpy.float64（np.float64 是 Python float 的子类，issubclass(np.float64, float) == True）
    # 因为 inverse_transform 直接从 ndarray 中取元素，ndarray 单元素类型为 numpy 标量而非 Python 内置 float
    # NumPy >= 1.25 起，numpy 标量的 repr 由 1.0 改为 np.float64(1.0)，故打印显示 np.float64 字样
    print(item)  # 打印单个样本的键值对；值类型为 np.float64（行为与 float 完全一致，仅 repr 显示不同）


特征DataFrame:
    城市=上海  城市=北京  城市=深圳     温度    湿度  风力=中  风力=弱  风力=强
0    0.0    1.0    0.0  100.0  60.0   0.0   0.0   1.0
1    1.0    0.0    0.0   90.0  70.0   1.0   0.0   0.0
2    0.0    0.0    1.0  110.0  80.0   0.0   1.0   0.0
特征名称: ['城市=上海' '城市=北京' '城市=深圳' '温度' '湿度' '风力=中' '风力=弱' '风力=强']
特征矩阵形状: (3, 8)
特征类型 <class 'numpy.ndarray'>

转换回的字典数据:
{'城市=北京': np.float64(1.0), '温度': np.float64(100.0), '湿度': np.float64(60.0), '风力=强': np.float64(1.0)}
{'城市=上海': np.float64(1.0), '温度': np.float64(90.0), '湿度': np.float64(70.0), '风力=中': np.float64(1.0)}
{'城市=深圳': np.float64(1.0), '温度': np.float64(110.0), '湿度': np.float64(80.0), '风力=弱': np.float64(1.0)}


# 第2节 CountVectorizer（英文）—— 词袋模型

**作用**：将文本转为**词频矩阵**（词袋模型，Bag of Words），统计每个词在文档中出现的次数。

**处理流程**：
1. `fit`：遍历所有文档，建立全局词汇表（词 → 列索引）
2. `transform`：对每篇文档统计词频，生成稀疏矩阵

**特点**：
- 英文文本按空白/标点自动分词，中文需预先分词
- 默认过滤长度 ≤1 的 token，不区分大小写
- 返回**稀疏矩阵** `scipy.csr_matrix`，需调用 `.toarray()` 转为密集 ndarray

**核心参数**：
- `max_features`（int）：限制词汇表大小，只保留频率最高的前 N 个词
- `min_df`（int/float）：最小文档频率，过滤低频词（int=最少出现文档数，float=比例）
- `analyzer`（callable）：自定义分词函数，中文场景传入 `jieba.cut`


In [43]:
# CountVectorizer 英文示例：将英文文本转为词频（词袋）矩阵

# 准备三篇英文文档；类型：list[str]，每个 str 为一篇文档
english_texts = [  # 3 篇文档，涵盖 ML/DL/NLP 主题
    "Machine learning is a branch of artificial intelligence",  # 文档1
    "Deep learning is a method of machine learning",  # 文档2
    "Natural language processing is an important application of artificial intelligence"  # 文档3
]  # 列表右括号

# 默认按空白与标点分词，过滤长度 ≤1 的 token，不区分大小写，构建全局词汇表
english_count_vec = CountVectorizer()  # 参数均取默认值；可用 max_features 限制词汇表大小

# fit_transform：先统计全部词汇（fit），再对每篇文档统计词频（transform）
english_X = english_count_vec.fit_transform(english_texts)  # 返回稀疏矩阵；行=文档，列=词，值=词频

# get_feature_names_out()：按列索引顺序返回词汇表中所有词；类型 ndarray[str]
english_vocabulary = english_count_vec.get_feature_names_out()  # 返回值：ndarray[str]，长度=词汇表大小

# toarray()：将稀疏矩阵转为密集 ndarray，便于打印和索引；内存占用增大
english_X_dense = english_X.toarray()  # 返回值：ndarray，形状 (3, 词汇表大小)，元素为整数词频
english_X_df = pd.DataFrame(english_X_dense, columns=english_vocabulary)  # 将词频矩阵转为 DataFrame；行=文档序号，列=词汇，值=整数词频


print("英文词汇表:", english_vocabulary)  # 打印全部词汇；按字母顺序排列
print("英文词频DataFrame:\n", english_X_df)  # 打印词频DataFrame；每行对应一篇文档
print("英文矩阵形状:", english_X_dense.shape)  # tuple，如 (3, 15) 表示 3 文档、15 个词

print("\n文档-词条矩阵解释:")  # 打印说明标题
for i, doc in enumerate(english_texts):  # i 为文档索引(int)，doc 为原始文本(str)
    print(f"文档 {i+1}: {doc}")  # 打印第 i+1 篇文档的原文
    print("包含的词条:")  # 打印子标题
    for j, term in enumerate(english_vocabulary):  # j 为词在词汇表中的列号(int)，term 为词(str)
        if english_X_dense[i, j] > 0:  # 仅打印该文档中实际出现的词（词频 > 0）
            print(f"  - '{term}' 出现 {english_X_dense[i, j]} 次")  # 打印词及其出现次数
    print()  # 文档间打印空行分隔


英文词汇表: ['an' 'application' 'artificial' 'branch' 'deep' 'important'
 'intelligence' 'is' 'language' 'learning' 'machine' 'method' 'natural'
 'of' 'processing']
英文词频DataFrame:
    an  application  artificial  branch  deep  important  intelligence  is  \
0   0            0           1       1     0          0             1   1   
1   0            0           0       0     1          0             0   1   
2   1            1           1       0     0          1             1   1   

   language  learning  machine  method  natural  of  processing  
0         0         1        1       0        0   1           0  
1         0         2        1       1        0   1           0  
2         1         0        0       0        1   1           1  
英文矩阵形状: (3, 15)

文档-词条矩阵解释:
文档 1: Machine learning is a branch of artificial intelligence
包含的词条:
  - 'artificial' 出现 1 次
  - 'branch' 出现 1 次
  - 'intelligence' 出现 1 次
  - 'is' 出现 1 次
  - 'learning' 出现 1 次
  - 'machine' 出现 1 次
  - 'of' 出现 1 次

文档 2: De

In [34]:
# jieba 分词示例：演示精确模式分词，理解中文分词对 NLP 的必要性

# text：待分词的中文字符串；类型 str
text = "我爱北京天安门，天安门上太阳升。"  # 包含重复词「天安门」，便于观察分词边界

# cut_all=False 为精确模式（HMM + 词典），适合大多数自然语言处理场景
print("默认分词:")  # 打印标题；jieba 内部结合词典与 HMM（隐马尔可夫模型）进行分词
seg_list = jieba.cut(text, cut_all=False)  # 返回 generator（生成器），元素为 str 词语；惰性求值
print("默认模式: " + " ".join(seg_list))  # join 消耗生成器，将词语以空格连接成字符串后打印

# 全模式：cut_all=True，尽可能切出所有词，会产生重叠词，适合搜索场景
print("\n全模式分词:")  # 全模式会切出更多词但有冗余
seg_list = jieba.cut(text, cut_all=True)  # 返回 generator；同一字符位可被多个词覆盖
print("全模式: " + " ".join(seg_list))  # 用斜线分隔便于观察冗余

# 搜索引擎模式：在精确模式基础上再切分长词，提高召回率
print("\n搜索引擎模式:")  # 兼顾搜索引擎对短词的需求
seg_list = jieba.cut_for_search(text)  # 返回 generator；长词会被拆成多个短词
print("搜索引擎模式: " + " ".join(seg_list))  # 打印结果

# 自定义词典：强制将「天安门上」识别为一个词
print("\n添加自定义词典:")  # 演示用户词典功能
jieba.add_word("天安门上")  # 将「天安门上」加入词典，提高该词被整体识别的概率
seg_list = jieba.cut(text, cut_all=False)  # 重新分词，分词结果受词典影响
print("添加自定义词典后: " + " ".join(seg_list))  # 对比添加词典前后的差异

# 词性标注：为每个词标注词性（名词/动词/形容词等）
print("\n词性标注:")  # 词性标注是 NLP 中的基础任务
import jieba.posseg as pseg  # posseg 模块提供词性标注功能
words = pseg.cut(text)  # 返回 generator，元素为 (词, 词性) 的具名元组
print("词性标注结果:")  # 打印标题
for word, flag in words:  # word 为词(str)，flag 为词性代码(str)，如 n=名词 v=动词
    print(f"{word} ({flag})")  # 打印每个词及其词性


默认分词:
默认模式: 我 爱 北京 天安门 ， 天安门 上 太阳升 。

全模式分词:
全模式: 我 爱 北京 天安 天安门 ， 天安 天安门 门上 太阳 太阳升 。

搜索引擎模式:
搜索引擎模式: 我 爱 北京 天安 天安门 ， 天安 天安门 上 太阳 太阳升 。

添加自定义词典:
添加自定义词典后: 我 爱 北京 天安门 ， 天安门上 太阳升 。

词性标注:
词性标注结果:
我 (r)
爱 (v)
北京 (ns)
天安门 (ns)
， (x)
天安门上 (x)
太阳升 (nr)
。 (x)


In [35]:
# CountVectorizer 中文示例：演示未分词时整句被当作一个 token 的问题

# texts：三句中文文本；类型 list[str]，每个 str 代表一篇文档
texts = [  # 三句话将用于后续 CountVectorizer 和 TF-IDF 示例
    "机器学习是人工智能的一个分支",  # 文档1：机器学习相关
    "深度学习是机器学习的一种方法",  # 文档2：深度学习相关
    "自然语言处理是人工智能的重要应用"  # 文档3：NLP 相关
]  # 列表右括号

# 中文无空格分隔，默认 analyzer 无法切词，整句成为一个 token
count_vec = CountVectorizer()  # 默认 analyzer='word'，按空白/标点分割；中文句子不含空格

# 对三句未分词的中文做 fit_transform；每个文档只有一个 token（整句）
X = count_vec.fit_transform(texts)  # 返回稀疏矩阵，形状 (3, 3)；每行一篇，对角线为1

vocabulary = count_vec.get_feature_names_out()  # 词汇表中三个元素即三句原文；类型 ndarray[str]
X_dense = X.toarray()  # 转为密集矩阵；形状 (3,3)，每行仅在自己对应列为 1
df = pd.DataFrame(X_dense, columns=vocabulary)  # 将词频矩阵转为 DataFrame；列=三句原文（未分词时每句即一个 token）


print("词汇表:", vocabulary)  # 三列分别是三句原文，体现未分词的问题
print("词频DataFrame:\n", df)  # 对角矩阵，说明每篇文档与自身词表匹配
print("矩阵形状:", X_dense.shape)  # 输出 (3, 3)


词汇表: ['机器学习是人工智能的一个分支' '深度学习是机器学习的一种方法' '自然语言处理是人工智能的重要应用']
词频DataFrame:
    机器学习是人工智能的一个分支  深度学习是机器学习的一种方法  自然语言处理是人工智能的重要应用
0               1               0                 0
1               0               1                 0
2               0               0                 1
矩阵形状: (3, 3)


In [36]:
# 使用 jieba 作为 CountVectorizer 的自定义分词器，实现中文词级词频统计

print("\n使用jieba分词的CountVectorizer:")  # 打印标题，与上方未分词示例对比
# analyzer 参数接受可调用对象；min_df=2 保留至少出现在 2 篇文档中的词（过滤低频词）
count_vec_jieba = CountVectorizer(analyzer=lambda x: jieba.cut(x), min_df=2)  # lambda返回generator；min_df=2过滤仅出现1次的词
X_jieba = count_vec_jieba.fit_transform(texts)  # 返回稀疏矩阵；行=文档，列=高频词，值=词频
vocabulary_jieba = count_vec_jieba.get_feature_names_out()  # 经 min_df 过滤后的词汇表；类型 ndarray[str]
df_jieba = pd.DataFrame(X_jieba.toarray(), columns=vocabulary_jieba)  # 将 jieba 分词词频矩阵转为 DataFrame；列=经 min_df 过滤后的高频词

print("jieba分词词汇表:", vocabulary_jieba)  # 打印过滤后的词汇表；只含至少出现 2 次的词
print("jieba分词词频DataFrame:\n", df_jieba)  # 密集矩阵；每个元素为整数词频
print("jieba分词矩阵形状:", X_jieba.toarray().shape)  # tuple，如 (3, 5) 表示 3 文档、5 个高频词



使用jieba分词的CountVectorizer:
jieba分词词汇表: ['人工智能' '学习' '是' '机器' '的']
jieba分词词频DataFrame:
    人工智能  学习  是  机器  的
0     1   1  1   1  1
1     0   2  1   1  1
2     1   0  1   0  1
jieba分词矩阵形状: (3, 5)


# 第3节 TF-IDF —— 词频·逆文档频率权重

**作用**：在词袋模型基础上，对词频加权，突出「本文常见、全局稀少」的关键词，压制「所有文档均出现」的停用词（如"的""是"）。

**公式**：

$$TF\text{-}IDF(t,d) = TF(t,d) \times IDF(t)$$

- $TF(t,d)$：词 $t$ 在文档 $d$ 中的出现频率
- $IDF(t) = \log\dfrac{1+N}{1+df(t)} + 1$，$N$ 为总文档数，$df(t)$ 为含词 $t$ 的文档数
- sklearn 默认对每行进行 **L2 归一化**，使行向量长度为 1

**与 CountVectorizer 的区别**：
- CountVectorizer 输出整数词频；TfidfVectorizer 输出浮点 TF-IDF 权重（0~1 量级）
- TF-IDF 能更好地区分文档主题，是文本分类/聚类的常用特征


In [37]:
# TF-IDF 示例：用词频×逆文档频率权重突出「本文常见、全局稀少」的词

print("\n使用jieba分词的TF-IDF处理:")  # 打印标题；TF-IDF 比词袋更能体现词的区分度
# TfidfVectorizer 与 CountVectorizer 接口相同，但输出为浮点 TF-IDF 权重，每行已 L2 归一化
tfidf_vec_jieba = TfidfVectorizer(analyzer=lambda x: jieba.cut(x))  # 用 jieba 分词；无 min_df 限制
X_tfidf_jieba = tfidf_vec_jieba.fit_transform(texts)  # 返回稀疏矩阵；值为 TF-IDF 浮点权重（0~1左右）
tfidf_vocabulary_jieba = tfidf_vec_jieba.get_feature_names_out()  # 词汇表；类型 ndarray[str]
df_tfidf_jieba = pd.DataFrame(X_tfidf_jieba.toarray(), columns=tfidf_vocabulary_jieba)  # 将 TF-IDF 稀疏矩阵转为 DataFrame；列=词汇，值=浮点 TF-IDF 权重

print("jieba分词词汇表:", tfidf_vocabulary_jieba)  # 打印全部词汇（含低频词，因未设 min_df）
print("jieba分词TF-IDF特征DataFrame:\n", df_tfidf_jieba)  # 打印 TF-IDF 权重 DataFrame；每列对应一个词，值越大表示该词对本文越重要
print("jieba分词矩阵形状:", X_tfidf_jieba.toarray().shape)  # tuple，如 (3, 14)



使用jieba分词的TF-IDF处理:
jieba分词词汇表: ['一个' '一种' '人工智能' '分支' '处理' '学习' '应用' '方法' '是' '机器' '深度' '的' '自然语言' '重要']
jieba分词TF-IDF特征DataFrame:
          一个        一种     人工智能        分支        处理        学习        应用  \
0  0.474961  0.000000  0.36122  0.474961  0.000000  0.361220  0.000000   
1  0.000000  0.389555  0.00000  0.000000  0.000000  0.592533  0.000000   
2  0.000000  0.000000  0.33110  0.000000  0.435357  0.000000  0.435357   

         方法         是        机器        深度         的      自然语言        重要  
0  0.000000  0.280520  0.361220  0.000000  0.280520  0.000000  0.000000  
1  0.389555  0.230077  0.296267  0.389555  0.230077  0.000000  0.000000  
2  0.000000  0.257129  0.000000  0.000000  0.257129  0.435357  0.435357  
jieba分词矩阵形状: (3, 14)


# 第4节 归一化（Normalization）—— MinMaxScaler

**作用**：将每列特征线性映射到指定区间 $[a, b]$，消除量纲差异，防止大数值特征主导模型。

**公式**（默认映射到 $[0, 1]$）：

$$x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

**适用场景**：
- KNN、SVM、神经网络等对特征尺度敏感的算法
- 特征分布不服从正态分布，或存在明显上下界

**注意**：对异常值（极端值）敏感，异常值会使正常值集中在极小的区间内。

**核心参数**：
- `feature_range`（tuple，默认 `(0, 1)`）：目标映射区间 `[min, max]`
- `copy`（bool，默认 `True`）：`False` 时在原数组上原地变换


In [38]:
# MinMaxScaler 归一化示例：将每列特征线性缩放到指定区间 [min, max]

import numpy as np  # 导入 numpy 用于构造示例数组

print("归一化示例:")  # 打印标题
# X：形状 (4, 3) 的 ndarray；4 个样本，每个样本有 3 个特征，含正负值
X = np.array([  # np.array 将嵌套列表转为 ndarray；dtype 默认为 int
    [1, -1, 2],   # 样本1：第1特征=1，第2特征=-1，第3特征=2
    [2, 0, 0],    # 样本2
    [0, 1, -1],   # 样本3
    [5, 2, 0]     # 样本4
])  # 右括号，X.shape == (4, 3)
print("原始数据矩阵:\n", X)  # 打印原始特征矩阵
print("数据矩阵形状:", X.shape)  # 输出 (4, 3)

# feature_range=(0,1)（默认）：每列应用公式 (x - min) / (max - min) 线性映射到 [0,1]
min_max_scaler = MinMaxScaler()  # 参数 feature_range 默认 (0,1)；copy=True 不修改原数组
X_minmax = min_max_scaler.fit_transform(X)  # 返回 ndarray，与 X 同形状，值域 [0,1]
print("\nMinMaxScaler归一化后的数据(缩放到[0,1]):\n", X_minmax)  # 每列最小值→0，最大值→1

# feature_range=(-1,1)：适合以 0 为中心的模型（如部分神经网络、SVM）
min_max_scaler_custom = MinMaxScaler(feature_range=(-1, 1))  # 自定义缩放区间
X_minmax_custom = min_max_scaler_custom.fit_transform(X)  # 返回 ndarray，值域 [-1,1]
print("MinMaxScaler归一化后的数据(缩放到[-1,1]):\n", X_minmax_custom)  # 每列最小值→-1，最大值→1


归一化示例:
原始数据矩阵:
 [[ 1 -1  2]
 [ 2  0  0]
 [ 0  1 -1]
 [ 5  2  0]]
数据矩阵形状: (4, 3)

MinMaxScaler归一化后的数据(缩放到[0,1]):
 [[0.2        0.         1.        ]
 [0.4        0.33333333 0.33333333]
 [0.         0.66666667 0.        ]
 [1.         1.         0.33333333]]
MinMaxScaler归一化后的数据(缩放到[-1,1]):
 [[-0.6        -1.          1.        ]
 [-0.2        -0.33333333 -0.33333333]
 [-1.          0.33333333 -1.        ]
 [ 1.          1.         -0.33333333]]


# 第5节 标准化（Standardization）—— StandardScaler

**作用**：将每列特征变换为均值为 0、标准差为 1 的分布（近似标准正态分布），适用于假设数据服从正态分布的算法。

**公式**：

$$x_{\text{std}} = \frac{x - \mu}{\sigma}$$

其中 $\mu$ 为列均值，$\sigma$ 为列总体标准差（分母为 $n$）。

**归一化 vs 标准化**：

| 方法 | 输出范围 | 对异常值 | 适用算法 |
|------|---------|---------|--------|
| MinMaxScaler | 固定区间 [0,1] | 敏感 | KNN、神经网络 |
| StandardScaler | 无固定范围，均值0方差1 | 较稳健 | 线性模型、SVM、PCA |

**核心属性**（fit 后）：
- `mean_`（ndarray）：各列均值
- `scale_`（ndarray）：各列总体标准差，等于 `sqrt(var_)`
- `var_`（ndarray）：各列总体方差


In [39]:
# StandardScaler 标准化：每列减均值再除标准差，结果近似服从 N(0,1)

print("\nStandardScaler标准化示例:")  # 打印标题
# X_example：3 个样本 × 3 个特征的浮点矩阵
X_example = np.array([[1., -1., 3.],   # 样本1；使用浮点数 1. 确保 dtype 为 float64
                      [2., 4., 2.],    # 样本2
                      [4., 6., -1.]])  # 样本3
print("原始数据矩阵:\n", X_example)  # 打印原始值

# with_mean=True（默认）减均值；with_std=True（默认）除标准差
std_scaler = StandardScaler()  # 初始化；fit 后生成 mean_ 和 scale_ 属性
X_std = std_scaler.fit_transform(X_example)  # 在 X_example 上估计均值/方差并立即变换；返回 ndarray
print("StandardScaler标准化后的数据:\n", X_std)  # 每列均值≈0，标准差≈1
# std_scaler.mean_：ndarray，形状 (n_features,)，dtype=float64
# fit 阶段对每一列（特征）计算总体均值（分母为样本数 n，非贝塞尔校正）
# 标准化公式中减去该均值：X_std = (X - mean_) / scale_
print("标准化前的均值:", std_scaler.mean_)  # 返回值：ndarray[float64]，每个元素对应一个特征列的拟合均值

# X_std.mean(axis=0)：沿 axis=0（样本轴）对标准化后矩阵逐列求均值
# 参数 axis：int，0 表示沿行方向折叠（即对每列的所有行取均值），返回形状 (n_features,)
# 理论值应为 0；浮点运算存在舍入误差，实际结果为接近 0 的极小数（约 1e-16 量级）
print("标准化后的均值:", X_std.mean(axis=0))  # 返回值：ndarray[float64]，各列均值；理论值全为 0

# std_scaler.scale_：ndarray，形状 (n_features,)，dtype=float64
# 等于各列总体标准差（sqrt(var_)），fit 阶段估计；标准化公式中作除数
# 若某列方差为 0（常数列），scale_ 对应元素为 1 以避免除零
print("标准化前的标准差:", std_scaler.scale_)  # 返回值：ndarray[float64]，每个元素为对应特征列的总体标准差

# X_std.std(axis=0)：沿样本轴对标准化后矩阵逐列计算标准差
# 参数 axis：int，0 表示对每列的所有行聚合；默认 ddof=0（总体标准差，与 sklearn 一致）
# 理论值应为 1.0；numpy 默认 ddof=0，与 StandardScaler 的总体标准差口径匹配
print("标准化后的标准差:", X_std.std(axis=0))  # 返回值：ndarray[float64]，各列总体标准差；理论值全为 1.0

# std_scaler.var_：ndarray，形状 (n_features,)，dtype=float64
# fit 阶段对每列计算总体方差（分母为 n），scale_ = sqrt(var_)
# 与 numpy 的 np.var(X_example, axis=0) 结果一致（ddof=0）
print("标准化前的方差:", std_scaler.var_)  # 返回值：ndarray[float64]，每个元素为对应特征列的总体方差

# X_std.var(axis=0)：沿样本轴对标准化后矩阵逐列计算方差
# 参数 axis：int，0 表示对每列的所有行聚合；默认 ddof=0（总体方差）
# 理论值应为 1.0；等于 std^2，与 StandardScaler 的计算口径匹配
print("标准化后的方差:", X_std.var(axis=0))  # 返回值：ndarray[float64]，各列总体方差；理论值全为 1.0




StandardScaler标准化示例:
原始数据矩阵:
 [[ 1. -1.  3.]
 [ 2.  4.  2.]
 [ 4.  6. -1.]]
StandardScaler标准化后的数据:
 [[-1.06904497 -1.35873244  0.98058068]
 [-0.26726124  0.33968311  0.39223227]
 [ 1.33630621  1.01904933 -1.37281295]]
标准化前的均值: [2.33333333 3.         1.33333333]
标准化后的均值: [-1.48029737e-16  7.40148683e-17  7.40148683e-17]
标准化前的标准差: [1.24721913 2.94392029 1.69967317]
标准化后的标准差: [1. 1. 1.]
标准化前的方差: [1.55555556 8.66666667 2.88888889]
标准化后的方差: [1. 1. 1.]


In [40]:
# SimpleImputer 示例：用统计量或常数填充 NaN 缺失值

print("\nSimpleImputer示例:")  # 打印标题
# X_missing：含 np.nan 的 (4,3) 矩阵；nan 代表缺失值
X_missing = np.array([  # 用嵌套列表构造浮点矩阵；np.nan 会使 dtype 自动变为 float64
    [np.nan, 2, 3],    # 第0列缺失
    [4, np.nan, 6],    # 第1列缺失
    [7, 8, np.nan],    # 第2列缺失
    [np.nan, np.nan, 6]  # 第0、1列均缺失
])  # 右括号；X_missing.shape == (4, 3)
print("包含缺失值的原始数据矩阵:\n", X_missing)  # 打印含 nan 的原始矩阵

# strategy='mean'：用该列非缺失值的算术均值填充；对正态分布特征效果好
mean_imputer = SimpleImputer(missing_values=np.nan, strategy='mean')  # missing_values 指定缺失标记
X_imputed_mean = mean_imputer.fit_transform(X_missing)  # 返回无 nan 的 ndarray，形状与原矩阵相同
print("\n使用均值填充后的数据矩阵:\n", X_imputed_mean)  # 查看填充结果

# strategy='median'：用中位数，比均值对异常值更稳健
median_imputer = SimpleImputer(strategy='median')  # 默认 missing_values=np.nan
X_imputed_median = median_imputer.fit_transform(X_missing)  # 返回无 nan 的 ndarray
print("\n使用中位数填充后的数据矩阵:\n", X_imputed_median)  # 与均值填充结果对比

# strategy='constant'：用 fill_value 指定的固定值填充；适合标记缺失（如填 -999）
constant_imputer = SimpleImputer(strategy='constant', fill_value=0)  # fill_value=0 即用 0 填充
X_imputed_constant = constant_imputer.fit_transform(X_missing)  # 返回无 nan 的 ndarray
print("\n使用常数0填充后的数据矩阵:\n", X_imputed_constant)  # 缺失位置全变为 0.0

# strategy='most_frequent'：用每列出现次数最多的值（众数）填充
most_frequent_imputer = SimpleImputer(strategy='most_frequent')  # 对类别特征（如字符串列）也适用
X_imputed_most_frequent = most_frequent_imputer.fit_transform(X_missing)  # 返回无 nan 的 ndarray
print("\n使用最频繁值填充后的数据矩阵:\n", X_imputed_most_frequent)  # 查看众数填充结果



SimpleImputer示例:
包含缺失值的原始数据矩阵:
 [[nan  2.  3.]
 [ 4. nan  6.]
 [ 7.  8. nan]
 [nan nan  6.]]

使用均值填充后的数据矩阵:
 [[5.5 2.  3. ]
 [4.  5.  6. ]
 [7.  8.  5. ]
 [5.5 5.  6. ]]

使用中位数填充后的数据矩阵:
 [[5.5 2.  3. ]
 [4.  5.  6. ]
 [7.  8.  6. ]
 [5.5 5.  6. ]]

使用常数0填充后的数据矩阵:
 [[0. 2. 3.]
 [4. 0. 6.]
 [7. 8. 0.]
 [0. 0. 6.]]

使用最频繁值填充后的数据矩阵:
 [[4. 2. 3.]
 [4. 2. 6.]
 [7. 8. 6.]
 [4. 2. 6.]]


# 第6节 方差阈值特征选择 —— VarianceThreshold

**作用**：过滤方差低于指定阈值的特征列。方差越小说明该特征在样本间几乎不变，对模型区分能力贡献极小。

**核心思路**：
- 方差为 0 的列：所有样本取值完全相同（常数特征），完全无区分能力
- 方差很小的列：取值变化极小，携带的信息量少

**公式**：

$$\text{Var}(X) = \frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2 > \text{threshold}$$

**核心参数**：
- `threshold`（float，默认 `0.0`）：方差阈值，仅保留**严格大于**该值的特征列

**核心方法**：
- `fit_transform(X)`：返回过滤后的 ndarray，形状 `(n_samples, 保留列数)`
- `get_support(indices=False)`：`indices=True` 返回保留列的原始下标 ndarray；`False` 返回布尔掩码


In [41]:
# VarianceThreshold 特征选择：删除方差低于阈值的特征列（几乎不变的特征没有区分能力）

print("\nVarianceThreshold示例:")  # 打印标题

# X_variance：3 个样本、4 个特征；第0、3列完全相同（方差为0），第2列变化最大
X_variance = np.array([  # 构造极端示例，使方差差异明显
    [0, 2, 0, 3],  # 样本1；第0列全0，第3列全3
    [0, 1, 4, 3],  # 样本2
    [0, 1, 1, 3]   # 样本3
])  # 右括号；X_variance.shape == (3, 4)
print("原始数据矩阵:\n", X_variance)  # 打印原始矩阵

feature_variances = np.var(X_variance, axis=0)  # axis=0 沿样本轴对每列求总体方差；返回 ndarray[float]，长度=列数
print("\n各特征的方差:", feature_variances)  # 第0、3列方差=0；第2列方差最大

# threshold=0.8：保留方差 > 0.8 的列；此处只有第2列满足条件
selector = VarianceThreshold(threshold=0.8)  # threshold 为方差下界；低于此值的列被删除
X_selected = selector.fit_transform(X_variance)  # 返回 ndarray，仅保留高方差列；形状 (3, 保留列数)

print("\n保留的特征索引:", selector.get_support(indices=True))  # 返回 ndarray[int]，保留列的原始下标
print("保留的特征方差:", feature_variances[selector.get_support()])  # 用布尔掩码索引方差数组
print("\n特征选择后的数据矩阵:\n", X_selected)  # 查看仅含高方差列的矩阵

selector_low = VarianceThreshold(threshold=0.1)  # 更低阈值会保留更多列
X_selected_low = selector_low.fit_transform(X_variance)  # 返回保留更多列的 ndarray
print("\n阈值为0.1时保留的特征索引:", selector_low.get_support(indices=True))  # 对比不同阈值的结果
print("阈值为0.1时特征选择后的数据矩阵:\n", X_selected_low)  # 打印结果



VarianceThreshold示例:
原始数据矩阵:
 [[0 2 0 3]
 [0 1 4 3]
 [0 1 1 3]]

各特征的方差: [0.         0.22222222 2.88888889 0.        ]

保留的特征索引: [2]
保留的特征方差: [2.88888889]

特征选择后的数据矩阵:
 [[0]
 [4]
 [1]]

阈值为0.1时保留的特征索引: [1 2]
阈值为0.1时特征选择后的数据矩阵:
 [[2 0]
 [1 4]
 [1 1]]


# 第7节 PCA —— 主成分分析（降维）

**作用**：通过正交线性变换，将高维数据投影到方差最大的若干正交方向（主成分），实现**无监督降维**，同时尽可能保留原始信息。

**核心步骤**：
1. 对数据**中心化**（减均值）
2. 计算**协方差矩阵**，对其进行特征值分解（或等价的 SVD 分解）
3. 取方差最大的前 $k$ 个特征向量作为新坐标轴
4. 将原始数据投影到新坐标系，得到 $k$ 维表示

**信息守恒**：降维后各主成分方差之和 = 原始数据总方差（信息量守恒）。

**核心参数**：
- `n_components`（int）：保留的主成分数量
- `n_components`（float，0~1）：自动选择使累计方差解释比例 ≥ 该值的最少主成分数

**核心属性**（fit 后）：
- `explained_variance_ratio_`（ndarray）：每个主成分解释的方差占总方差的比例
- `components_`（ndarray）：主成分方向（特征向量），形状 `(n_components, n_features)`


In [42]:
# PCA 主成分分析：通过封装函数演示降维、方差守恒与解释比例

def understand_pca(data, n_components=None):  # 教学封装函数：对输入数据执行 PCA 并打印方差信息；参数 data 为 ndarray，n_components 控制保留维度
    """
    教学用 PCA 封装：拟合数据并打印方差信息。

    参数:
        data: ndarray，形状 (n_samples, n_features)，输入数据矩阵
        n_components: int 时保留指定数量主成分；float∈(0,1) 时保留累计方差达该比例的主成分；默认 None 保留全部

    返回:
        pca: 已拟合的 PCA 对象，含 explained_variance_ratio_ 等属性
        transformed_data: ndarray，形状 (n_samples, n_components_实际)
        explained_variance_ratio_: ndarray，各主成分解释的方差占比
    """
    import numpy as np  # 函数内导入，保证可独立运行
    import matplotlib.pyplot as plt  # 导入绘图库（本函数暂未使用，作为扩展预留）

    pca = PCA(n_components=n_components)  # 创建 PCA 模型；n_components 控制降维目标
    transformed_data = pca.fit_transform(data)  # 先 SVD 分解估计主方向，再将数据投影到主成分空间

    print("原始数据:\n", data)  # 打印输入矩阵
    print("\n数据形状:", data.shape)  # tuple，如 (3, 4) 表示 3 个样本、4 个特征

    print("\nPCA转换后的数据:\n", transformed_data)  # 打印降维后的坐标；列是主成分轴方向
    print("\n转换后数据形状:", transformed_data.shape)  # tuple，如 (3, 2) 表示降到 2 维
    print("\n转换后数据方差:", np.var(transformed_data, axis=0))  # 各主成分方向的方差；第1主成分最大
    print("\n转换后数据方差和:", np.var(transformed_data, axis=0).sum())  # 应≈原始数据总方差（信息守恒）

    print("\n各主成分解释的方差比例:", pca.explained_variance_ratio_)  # ndarray；各元素之和=1（全保留时）
    print("累计解释的方差比例:", np.sum(pca.explained_variance_ratio_))  # 累计方差比；全保留时=1.0

    return pca, transformed_data, pca.explained_variance_ratio_  # 返回模型、降维数据、方差比例

# 构造 3 个样本、4 个特征的示例矩阵；行=样本，列=特征
X = np.array([[2, 8, 4, 5],   # 样本1
              [6, 3, 0, 8],   # 样本2
              [5, 4, 9, 1]])  # 样本3

print("\n原始数据方差:", np.var(X, axis=0))  # 每列方差；反映各特征的离散程度
print("\n原始数据方差和:", np.var(X, axis=0).sum())  # 总方差；PCA 后应保持不变

print("完整PCA示例:")  # 不降维，保留全部主成分
pca_full, data_full, var_ratio_full = understand_pca(X)  # 返回：PCA对象、(3,3)矩阵、方差比例数组

print("\n\n降维到2个主成分的PCA示例:")  # 演示指定累计方差比例的降维
# n_components=0.9：自动选择能解释 ≥90% 总方差的最少主成分数量
pca_2d, data_2d, var_ratio_2d = understand_pca(X, n_components=0.9)  # 返回：PCA对象、(3,2)矩阵、方差比例数组



原始数据方差: [ 2.88888889  4.66666667 13.55555556  8.22222222]

原始数据方差和: 29.333333333333336
完整PCA示例:
原始数据:
 [[2 8 4 5]
 [6 3 0 8]
 [5 4 9 1]]

数据形状: (3, 4)

PCA转换后的数据:
 [[-1.28620952e-15  3.82970843e+00  5.26052119e-16]
 [-5.74456265e+00 -1.91485422e+00  5.26052119e-16]
 [ 5.74456265e+00 -1.91485422e+00  5.26052119e-16]]

转换后数据形状: (3, 3)

转换后数据方差: [2.20000000e+01 7.33333333e+00 6.48230758e-63]

转换后数据方差和: 29.333333333333332

各主成分解释的方差比例: [7.50000000e-01 2.50000000e-01 9.43400565e-33]
累计解释的方差比例: 1.0


降维到2个主成分的PCA示例:
原始数据:
 [[2 8 4 5]
 [6 3 0 8]
 [5 4 9 1]]

数据形状: (3, 4)

PCA转换后的数据:
 [[-1.28620952e-15  3.82970843e+00]
 [-5.74456265e+00 -1.91485422e+00]
 [ 5.74456265e+00 -1.91485422e+00]]

转换后数据形状: (3, 2)

转换后数据方差: [22.          7.33333333]

转换后数据方差和: 29.333333333333332

各主成分解释的方差比例: [0.75 0.25]
累计解释的方差比例: 1.0
